# Kv2.1 F412L S6-neighborhood distance analysis

Default ensemble subset: **allOk3 + structural/interface QC** (`all_ok_3_structural_interface_qc`; corrected 3 Å convergence, G377 tetramer-integrity filtering, and a 27 Å pore–VSD interface cutoff).
The same residue pairs are used for four complementary comparisons: WT versus F412L within each protocol, and vanilla versus masked within each construct. The 8SD3 WT and 8SDA L403A structures are overlaid as experimental state references because L403A and F412L were proposed to perturb related S6 rearrangements. These references test whether F412L samples an L403A-like direction; 8SDA is not a mutation-matched F412L structure.


In [ ]:
from pathlib import Path
import sys
import pandas as pd
repo_root = Path.cwd() if (Path.cwd() / 'shared').is_dir() else Path.cwd().parent
sys.path.insert(0, str(repo_root)) if str(repo_root) not in sys.path else None
from shared.mutation_site_analysis import nearby_distance_columns, rank_nearby_shifts, top_aliases
import importlib, shared.dataset_selection as shared_dataset_selection
importlib.reload(shared_dataset_selection)
from shared.dataset_selection import distance_csv_options, load_selected_distance_csv
from shared.plotting import KV21_PALETTE, apply_kv21_style, plot_split_ensemble_with_experimentals, plot_protocol_split_with_experimentals
apply_kv21_style()


In [ ]:
DATASET_SELECTION = 'all_ok_3_structural_interface_qc'  # Persisted convergence, tetramer, and interface QC.
data = repo_root / 'kv21' / 'dataDistances'
DATASET_PATHS = {
    'WT vanilla': distance_csv_options(repo_root, data / '26-02-11_Kv2.1_wt_vanillaAF2_distances_all.csv', 'Kv21', 'WT', 'vanilla'),
    'WT masked': distance_csv_options(repo_root, data / '26-02-11_Kv2.1_wt_maskedAF2_distances_all.csv', 'Kv21', 'WT', 'masked'),
    'F412L vanilla': distance_csv_options(repo_root, data / '26-02-11_Kv2.1_f412l_vanillaAF2_distances_all.csv', 'Kv21', 'F412L', 'vanilla'),
    'F412L masked': distance_csv_options(repo_root, data / '26-02-11_Kv2.1_f412l_maskedAF2_distances_all.csv', 'Kv21', 'F412L', 'masked'),
}
DATASET_PATHS
wt_van = load_selected_distance_csv('WT vanilla', DATASET_PATHS['WT vanilla'], DATASET_SELECTION)
wt_mask = load_selected_distance_csv('WT masked', DATASET_PATHS['WT masked'], DATASET_SELECTION)
mut_van = load_selected_distance_csv('F412L vanilla', DATASET_PATHS['F412L vanilla'], DATASET_SELECTION)
mut_mask = load_selected_distance_csv('F412L masked', DATASET_PATHS['F412L masked'], DATASET_SELECTION)


In [ ]:
# Select one compact set of F412-adjacent distances for every panel. Ranking by the
# largest shift seen in either protocol avoids changing the x axis between comparisons.
vanilla_shifts = rank_nearby_shifts(wt_van, mut_van, nearby_distance_columns(wt_van, 412))
masked_shifts = rank_nearby_shifts(wt_mask, mut_mask, nearby_distance_columns(wt_mask, 412))
combined_shifts = pd.concat([vanilla_shifts, masked_shifts], ignore_index=True)
combined_shifts = combined_shifts[combined_shifts['distance'].str.startswith('CA_CA_')]
combined_shifts = combined_shifts.sort_values('abs_shift_A', ascending=False).drop_duplicates('distance')
aliases = top_aliases(combined_shifts, n=12, residue_number_offset=-2)
display(vanilla_shifts.head(12).assign(protocol='vanilla'))
display(masked_shifts.head(12).assign(protocol='masked'))

# Calculate the matching Cα distances directly from the experimental coordinates.
# Visible aliases already use the experimental/paper numbering (model numbering − 2).
import math, re
def read_ca(path):
    atoms = {}
    with open(path) as handle:
        for line in handle:
            if line.startswith('ATOM') and line[12:16].strip() == 'CA':
                atoms[(line[21], int(line[22:26]))] = tuple(float(line[i:i+8]) for i in (30, 38, 46))
    return atoms

def experimental_for_aliases(alias_map, pdb_path):
    atoms, distances, missing = read_ca(pdb_path), {}, []
    for alias in alias_map:
        residues = re.findall(r'([A-D])_([A-Z]{3})(\d+)', alias)
        if len(residues) != 2:
            missing.append(alias); continue
        first = atoms.get((residues[0][0], int(residues[0][2])))
        second = atoms.get((residues[1][0], int(residues[1][2])))
        if first is None or second is None:
            missing.append(alias); continue
        distances[alias] = [round(math.dist(first, second), 3)]
    return distances, missing

exp_wt, missing_wt = experimental_for_aliases(aliases, repo_root/'kv21'/'experimental'/'8SD3.pdb')
exp_l403a, missing_l403a = experimental_for_aliases(aliases, repo_root/'kv21'/'experimental'/'8SDA.pdb')
print(f'Experimental matches: 8SD3={len(exp_wt)}/{len(aliases)}, 8SDA={len(exp_l403a)}/{len(aliases)}')
if missing_wt or missing_l403a:
    print('Unresolved aliases:', sorted(set(missing_wt + missing_l403a)))
experimental_distances = [exp_wt, exp_l403a]
experimental_labels = ['Experimental | 8SD3 WT', 'Experimental | 8SDA L403A state reference']
experimental_colors = ['#D55E00', '#0072B2']
region = 'F412-adjacent S6 neighborhood'

# Mutation effect within each prediction protocol.
plot_split_ensemble_with_experimentals(wt_van, mut_van, aliases, 'F412L', 'vanilla', region, experimental_distances, experimental_labels, experimental_colors)
plot_split_ensemble_with_experimentals(wt_mask, mut_mask, aliases, 'F412L', 'masked', region, experimental_distances, experimental_labels, experimental_colors)

# Masking effect within each sequence background.
plot_protocol_split_with_experimentals(wt_van, wt_mask, aliases, 'Kv2.1 | WT | vanilla vs masked | F412-adjacent S6 neighborhood | experimental state references', [KV21_PALETTE['WT_VAN'], KV21_PALETTE['WT_HM']], experimental_distances, experimental_labels, experimental_colors)
plot_protocol_split_with_experimentals(mut_van, mut_mask, aliases, 'Kv2.1 | F412L | vanilla vs masked | F412-adjacent S6 neighborhood | experimental state references', [KV21_PALETTE['F412L_VAN'], KV21_PALETTE['F412L_HM']], experimental_distances, experimental_labels, experimental_colors)

## Contacts altered by F412L (model/CSV F414L)
from shared.mutation_contact_analysis import plot_mutation_contacts, plot_selected_contact_distributions
f412l_contact_table, _ = plot_mutation_contacts(
    {'vanilla': (wt_van, mut_van), 'masked': (wt_mask, mut_mask)},
    'PHE414A', 'LEU414A', 'F412L',
    {'vanilla': KV21_PALETTE['F412L_VAN'], 'masked': KV21_PALETTE['F412L_HM']}, channel='Kv2.1',
)
display(f412l_contact_table.head(15).round(3))
f412l_top_partners = list(dict.fromkeys(f412l_contact_table['Partner']))[:3]
plot_selected_contact_distributions(
    {'vanilla': (wt_van, mut_van), 'masked': (wt_mask, mut_mask)},
    'PHE414A', 'LEU414A', 'F412L', f412l_top_partners,
    {'vanilla': (KV21_PALETTE['WT_VAN'], KV21_PALETTE['F412L_VAN']), 'masked': (KV21_PALETTE['WT_HM'], KV21_PALETTE['F412L_HM'])}, channel='Kv2.1',
)
